<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="http://www.uoc.edu/portal/_resources/common/imatges/marca_UOC/UOC_Masterbrand.jpg", align="left">
</div>
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">Named Entity Linking</p>
    <p style="margin: 0; text-align:right;">Màster universitari de Ciència de Dades (<i>Data Science</i>)</p>
<p style="margin: 0; text-align:right; padding-button: 100px;">Estudis d'Informàtica, Multimèdia i Telecomunicació</p>
</div>
</div>
<div style="width: 100%; clear: both;">
<div style="width:100%;">&nbsp;</div>

#Named Entity Linking

**Autors:** Nadjet Bouayad-Agha & Josep Maria Sabater

<a name="introduccio"></a>**Objectiu Notebook**

'REFERENCIAR Entitats Anomenades' ('Named Entity Linking' o 'NEL' per les seves sigles en anglès) d'un text és la tasca NLP orientada a assignar a cada `'entitat anomenada'` (1) del text una entrada a una base de coneixement (KB) de referència: `'entitat de la KB'`. 

NEL, per tant, pressuposa l'existència d'una KB de referència. La KB acostuma a dependre de la naturalesa del treball que requereix NEL. Una KB de propòsit general sovint utilitzada és la [Wikipedia](https://ca.wikipedia.org/wiki/Portada) (en la versió de l'idioma del text o bé contemplant una visió multilingüe de les diferents versions de Wikipedia segons idioma) o algun dels projectes paral·lels associats: [Wikidata](https://www.wikidata.org/wiki/Wikidata:Main_Page), [DBPedia](https://www.dbpedia.org/). 

En endavant, s'empren els termes:

*   `'Menció'`: token o tokens de text detectats per una tasca NER o de Spotting o matching de tokens.
*   `'Entitat de la KB'` o `'Entitat'`: entrada de la KB que referencia una `menció`.

La majoria de mètodes NEL solen contemplar dues etapes:

1.   Generar `entitats` candidates de la KB per a cada `menció`. 
2.   Obtenir un rànquing de les `entitats` candidates de cada menció que permeti seleccionar l'`entitat` de la KB corresponent a la menció

En aquest notebook,  primer presentem una API de NEL acabat, concretament, [DBPedia Spotlight](https://www.dbpedia-spotlight.org/).  

Després presentem com crear una KB i entrenar un model amb el mòdul Entity Linker de [spaCy](https://spacy.io/api). 

**Organització del notebook**

El notebook s'estructura en les seccions següents:

* [Secció 1](#install) instal·la i importa les llibreries i models necessaris per a l'estudi.  

* [Secció 2](#DBPedia) Entity Linking amb [DBPedia Spotlight](https://www.dbpedia-spotlight.org/). Mètode estadístic en pipeline que té com a base de coneixement de referència KB [DBPedia](https://www.dbpedia.org/). [DBPedia Spotlight](https://www.dbpedia-spotlight.org/) es defineix com a *'It is a tool for automatically annotating mentions of DBpedia resources in text, providing a solution for linking unstructured information sources to the Linked Open Data cloud through DBpedia'* (2).

* [Secció 3](#spaCy) entrena un model de [spaCy per a Entity Linking](https://spacy.io/api/entitylinker) amb una mini-KB.

<p>&nbsp</p>  
(1) Vegeu el notebook previ 'Mòdul NER.ipynb'$$$$  
(2) Font: [DBPedia Spotlight](https://www.dbpedia-spotlight.org/). Data visita: 20/10/2021

# <a name="install"></a>1 Importar llibreries i models

## 1.1 Instal·lar spaCy i altres llibreries

Es força la instal·lació de la versió spaCy 3.2.0 ('Google Colab' proporciona per defecte la versió 2.2.4 a la data de finalització d'aquest notebook, desembre de 2021)

Nota: Consultar [aquí](https://spacy.io/usage#changelog) les diferents versions de spaCy.


In [ ]:
# Instal·lació spacy
!pip install spacy==3.2.0
#!pip install spacy-lookups-data

     |████████████████████████████████| 6.0 MB 4.6 MB/s 
     |████████████████████████████████| 181 kB 54.8 MB/s 
     |████████████████████████████████| 653 kB 47.2 MB/s 
     |████████████████████████████████| 10.1 MB 44.6 MB/s 
     |████████████████████████████████| 457 kB 52.9 MB/s 
     |████████████████████████████████| 42 kB 1.3 MB/s 
     |████████████████████████████████| 58 kB 5.3 MB/s 
  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 4.2.0
    Uninstalling typing-extensions-4.2.0:
      Successfully uninstalled typing-extensions-4.2.0
  Attempting uninstall: catalogue
    Found existing installation: catalogue 1.0.0
    Uninstalling catalogue-1.0.0:
      Successfully uninstalled catalogue-1.0.0
  Attempting uninstall: srsly
    Found existing installation: srsly 1.0.5
    Uninstalling srsly-1.0.5:
      Successfully uninstalled srsly-1.0.5
  Attempting uninstall: smart-open
    Found existing installation: smart-open 6.0.0
    U

Es força la instal·lació de la versió del model spaCy per a català 'ca_core_news_lg' compatible amb la versió spaCy 3.2.0. 

Nota: La compatibilitat es pot obtenir de la descripció [spaCy dels models en català](https://spacy.io/models/ca), opció 'release details' de cada model. La llista completa de compatibilitats es pot veure [aquí](https://github.com/explosion/spacy-models/blob/master/compatibility.json)

In [ ]:
# Càrrega del model spaCy en català: ca_core_news_lg
!python -m spacy download ca_core_news_lg-3.2.0 --direct

     |████████████████████████████████| 575.1 MB 12 kB/s 
✔ Download and installation successful
You can now load the package via spacy.load('ca_core_news_lg')


In [ ]:
# Validar la compatibilitat de les versions
!python -m spacy validate

✔ Loaded compatibility table

================= Installed pipeline packages (spaCy v3.2.0) =================
ℹ spaCy installation: /usr/local/lib/python3.7/dist-packages/spacy

NAME              SPACY            VERSION                            
ca_core_news_lg   >=3.2.0,<3.3.0   3.2.0   ✔



## 1.2 Importar spaCy i altres llibreries

In [ ]:
import spacy
print (f"Spacy version installed: {spacy.__version__}")
from spacy.kb import KnowledgeBase               # KB de spaCy
from spacy.training import Example
from spacy.ml.models import load_kb
from spacy.util import minibatch, compounding

Spacy version installed: 3.2.0


In [ ]:
# Accés a Google Drive
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

In [ ]:
import requests            # Per a la crida a l'API DBPedia Spotlight
import json
from pathlib import Path
import csv   
import random  
from collections import Counter    
from sklearn.model_selection import train_test_split         

In [ ]:
# Directori de treball de models i fitxers necessaris per a aquest notebook
%cd /gdrive/My Drive/ner_nel_material/resources
print (f"Current working directory: {Path.cwd()}")

# <a name="DBPedia"></a>2 Entity Linking amb DBPedia Spotlight

[DBPedia Spotlight](https://www.dbpedia-spotlight.org/) fa servir DBPedia com a KB de referència. 

Consta de les etapes següents (per a més detalls, vegeu article [aquí](http://jodaiber.de/doc/entity.pdf)):
*   'Spotting': detecció de `mencions` (seqüències de tokens) en el text que corresponen a una entrada DBPedia.  Això ho fa amb string matching.
*   'Candidate Selection': selecció d'`entitats candidates` de la KB per a cada menció.  
* 'Filtering': Filtratge d'entitats candidates en funció de requeriments d'usuari.
*   'Disambiguation': selecció de la millor entitat candidata per a cada menció.

Procés DBPedia Spotlight:  
<p>&nbsp</p>


![spotlight](https://www.dbpedia-spotlight.org/images/ipad.jpg)
<p>&nbsp</p>
(Font: https://www.dbpedia-spotlight.org/) 

DBPedia Spotlight admet els següents tipus d'accés:

*   [Demo O/L](https://demo.dbpedia-spotlight.org/).
*   [In-house Server](https://github.com/dbpedia-spotlight/dbpedia-spotlight-model#run-your-own-server). Preferible quan es precisa garantir la disponibilitat de servei i millora el temps de resposta.
* [Accés API](https://www.dbpedia-spotlight.org/api/ca). (Si es desitja utilitzar un altre idioma de DBPedia, substituir a l'URL el component 'ca', català, per un altre codi d'idioma)


## 2.1 Accés amb API de Python


Es mostra a continuació un exemple d'Entity Linking utilitzant l'API de DBPedia Spotlight.

* URL d'accés a l'API, DBPedia anglès: https://www.dbpedia-spotlight.org/api o bé https://www.dbpedia-spotlight.org/api/en
* URL DBPedia català: https://www.dbpedia-spotlight.org/api/ca. Per a altres idiomes, substituir [ca] per acrònim en un altre idioma.

In [ ]:
def dbpedia_spotlight_access (url, text, confidence, support, list_types):
  """Acces to DBPedia Spotlight service.
    
    Parameters:
      url (str): url DBPedia Spotlight API
      text (str): Text for Entity Linking
      confidence (float): 'confidence score for disambiguation / linking'
      support (int): 'how prominent is this entity, i.e. number of inlinks in Wikipedia'
      list_types (list): list of types to filter according to the dbpedia ontology (rdf:type). List can be empty.
      
    Returns:
      json: DBPedia Spotlight return in json format.
  """

  dbpedia_tipus = ",".join([f"DBpedia:{x}" for x in list_types]) 
  params = {"text": text, 
            "confidence": confidence,     
            "support": support,           
            "types": dbpedia_tipus
            }
  # tipus de contingut de la resposta
  headers = {'accept': 'application/json'}
  # GET Request
  res = requests.get(url, params=params, headers=headers)
  if res.status_code != 200:
    raise Exception (f"Error API Spotlight. Error: {res.status_code}. Parameters: {params}")

  # A https://github.com/dbpedia-spotlight/dbpedia-spotlight/wiki/Web-service#candidates vegeu el significat dels camps de res.content
  return json.loads(res.content)


In [ ]:
# Funció de suport que retorna un text a imprimir en línies de 100 caràcters
def get_text_to_print(text):
  """Format given text.

    Parameters:
      text (str): text to print

    Returns:
      str: text formatted in 100 character lines with an initial line numbering the characters
  """
  line_length = 100
  line_poss   = "     1-------10--------20--------30--------40--------50--------60--------70--------80--------90-------100"
  text        = text.replace("\n", " ")     # Perquè el caràcter salt de línia no salti a l'imprimir text formatat
  text        = text.replace("\r", " ")     # En textos procedents de Viquipèdia s'ha detectat caràcter '\r' que s'interpretaria com a posicionar-se a l'inici de línia si no es canvia a blanc
  text_format = "\n".join([ f"{i//line_length:<5}{text[i:i+line_length]}"  for i in range(0, len(text), line_length) ])
  return line_poss + "\n" + text_format + "\n" + line_poss

In [ ]:
def do_nel(text, confidence=0.5, support=0, dbpedia_lan="ca", 
                         list_types =["Person", "Location", "Organisation", "Thing", "Device"], 
                         base_url="http://api.dbpedia-spotlight.org/", 
                         action="annotate"):
  """Run Entity Linking through DBPedia Spotlight. Print entities and their links to DBPedia (@URIs)

    Parameters:
      text (str): text for Entity Linking
      confidence (float): 'confidence score for disambiguation / linking'
      support (int): 'how prominent is this entity, i.e. number of inlinks in Wikipedia'
      dbpedia_lan (str): DBPedia language (ca, fr, en, ...)
      list_types (list): list of types to filter according to the dbpedia ontology (rdf:type). List can be empty.
      base_url: url DBPedia Spotlight API
      action: action defined in DBPedia Spotlight API (annotatate, spot, candidates)

    Returns: ---
  """
  url = base_url+dbpedia_lan+"/"+action

  print(f"URL: {url}") 

  # Accés a l'API Spotlight i imprimir per a comprovació
  dicc = dbpedia_spotlight_access (url, text, confidence, support, list_types)

  print ("Given the text:\n")
  print (get_text_to_print (dicc["@text"]))
  print (f"\nAPI DBPedia Spotlight result. DBPedia language: '{dbpedia_lan}'. Types filtered: '{list_types}':\n")
  print (f"{'Entity (ent.text)':<30}  Chr_start  Chr_end  {'URI DBPedia':<60}  {'DBPedia Type'}  ")
  print (f"{'=================':<30}  =========  =======  {'===========':<60}  {'============'}   ")

  if "Resources" not in dicc:
    print (f"***DBPedia Spotlight hasn't identified any entity in the text***")
  else:
    for mention in dicc["Resources"]:
      list_types = [x[8:] for x in mention["@types"].split(",") if x.startswith("DBpedia")]  
      print (f'{dicc["@text"][int(mention["@offset"]):int(mention["@offset"])+len(mention["@surfaceForm"])]:<30}  {int(mention["@offset"]):>9}  {int(mention["@offset"])+len(mention["@surfaceForm"]):>7}  {mention["@URI"]:<60}  {list_types}')
      
  print ("\n")


In [ ]:
corpus     = ["El Brexit es va dur a terme el 31 de gener del 2020 sota el govern de Boris Johnson. \
El negociador en cap de la Unió Europea (UE) va ser Michel Barnier. Menys de dos anys després, el Regne Unit exigeix canvis fonamentals \
que afecten el Protocol Nord-Irlandès, protocol que garanteix que no hi haurà una frontera dura entre la República d'Irlanda i Irlanda del Nord. \
Michel Barnier és un polític francès, va néixer el 9 de gener de 1951 a La Tronche (Delfinat).", 
              "La companyia Apple ven productes com ara iPhone o iPad",
              "Microsoft va ser fundada per Paul Allen i Bill Gates el 4 d'abril del 1975 a Califòrnia per desenvolupar i \
comercialitzar intèrprets de BASIC pel Altair 8800, un microordinador dissenyat el 1974 i basat en el processador Intel 8080.",
              "El general Prim, qui fou president del Consell de Ministres d'Espanya, va néixer a Reus (Tarragona), comarca del Baix Camp",
              "Hi ha un Jordi Pujol, president de la Generalitat de Catalunya, però no s'ha de confondre amb el traductor, autor de versions d'obres teatrals i professor de català Jordi Pujol Cofan.",
              "Gerard Piqué és jugador del Futbol Club Barcelona, club de futbol de Barcelona, Catalunya",
              "Gerard Piqué és jugador del Barça, Futbol Club Barcelona, club de futbol de Barcelona, Catalunya",
              ]

In [ ]:
do_nel(corpus[-2])        # si es prova sense filtre per tipus: 'do_nel(corpus[-2], list_types=[])', detecta més mencions, tot i que no corresponen a cap tipus DBPedia de la DBPedia en català

Podem provar amb NEL de l'Anglès:

In [ ]:
do_nel(corpus[-2], dbpedia_lan="en")

Veiem en el penúltim exemple (exemple índex -2) que "Gerard Piqué" és reconegut amb el DBPedia Spotlight de l'anglès, que en general té més conceptes. Dit això, mencions en català com "Catalunya" o "Unió Europea" (exemple índex 1) no apareixeran al DBPedia de l'anglès.

## 2.2 Exercicis



1.   Utilitza DBPedia Spotlight amb diferents combinacions dels textos proposats en el corpus exemple anterior i DBPedia amb diferents idiomes i tipus entitat. Comprova les diferències de resultats.
2. Prova amb textos similars on es mostri que el context influeix en la tasca NEL (com ara els exemples anteriors: "Gerard Piqué és jugador del Futbol Club Barcelona..." i "Gerard Piqué és jugador del Barça, Futbol Club Barcelona...".
3. Per a un mateix text utilitza, a més de DBPedia Spotlight, les eines/demos de NEL Wikifier i Babelfy (vegeu secció [Recursos](#recursos)). Comprova similituds i diferències.


#<a name="spaCy"></a>3 Entity Linking amb spaCy

Ara mostrem com entrenar un model d'Entity Linking amb Spacy, a nivell molt reduït. Aquest entrenament es basa en el següent procés:

1.   Crear la base de coneixement (KB) de spaCy amb `entitats` i `àlies` (un àlies pot ser-ho d'una o més entitats de la KB)
2. Afegir la tasca NEL "entity linker" al pipeline de spaCy.
3. Detectar `mencions` del text amb el component spaCy NER del seu pipeline (vegeu el nostre notebook sobre NER). Això vol dir que només podem detectar mencions identificades com a entitats anomenades.
4. Resoldre NEL entre les `entitats candidates` de cada `menció`, on les entitats candidates s'obtenen com les entitats associades a un `àlies` idèntic a la menció.

Per a aquest exercici, s'utilitzarà el model `ca_core_news_lg` de SpaCy per al català.


In [ ]:
# Càrrega del model 'ca_core_news_lg'
MODEL = "ca_core_news_lg"
nlp = spacy.load(MODEL)
print (f"Model '{MODEL}' loaded correctly!")

##3.1 Creació de la KB de spaCy

spaCy disposa de la classe [KnowledgeBase](https://spacy.io/api/kb) per a emmagatzemar la KB que s'ha d'utilitzar durant el procés Entity Linking. Aquesta KB conté:
* Entitats del domini de referència. Per a cada entitat: (i) el seu ID, (ii) el 'doc embedding' de la descripció de l'entitat corresponent al ID.
* Els 'àlies': token o tokens que poden referenciar una o més entitats.

La creació d'aquesta KB dependrà del domini específic de treball. 

Per a aquest exercici, es parteix de [Wikidata](https://www.wikidata.org/wiki/Wikidata:Main_Page) com a KB de referència, especialment per a extreure'n els identificadors (QID) de cada entitat, i [Viquipèdia](https://ca.wikipedia.org/wiki/Portada) per a les descripcions de les entitats.

En un cas real, es podria realitzar una descàrrega total de Wikidata, així com els [àlies](https://www.wikidata.org/wiki/Help:Aliases) per a cada entitat, addicionalment cal complementar aquests àlies amb altres fonts possibles (com ara considerar un àlies d'una entitat qualsevol menció que a Wikipedia enllaça amb aquesta entitat, 'anchor text',...).


No obstant això, donat el caràcter d'exemple d'aquest notebook, es restringeix la KB a una única menció, 'Piqué', i les tres entitats que podrien correspondre a aquesta menció a Viquipèdia i Wikidata:

*   [Gerard Piqué](https://www.wikidata.org/wiki/Q17507) amb QID: Q17507. Futbolista.
*   [Josep Piqué](https://www.wikidata.org/wiki/Q1366326) QID: Q1366326. Polític.
* [Elisabetta Piqué](https://www.wikidata.org/wiki/Q21074333) QID: Q21074333. Periodista.

El fitxer 'entitats_KB.csv' s'ha emplenat manualment. Conté els camps: QID, nom entitat i descripció de cadascuna de les tres entitats anteriors. S'utilitza aquest fitxer per a emplenar les entitats de la KB de spaCy. L'exemple seria generalitzable ampliant el nombre d'entitats presents a 'entitats_KB.csv'.

Accions que es realitzen:


*   Carregar el KB de spaCy a partir de 'entitats_KB.csv'.
* Completar el KB carregat amb els àlies de cada entitat.



In [ ]:
def get_ents_to_print (model, text):
  """Print text and its entities.

    Parameters:
      model (model spaCy): spaCy model used for entity recognition
      text (str): text for entity recogniton.

    Returns: 
      doc (spaCy 'doc' class): doc object from text
  """
  doc = model (text)
  print (f"\nModel '{model.meta['lang']+'_'+model.meta['name']}'' applied to text:\n\n{get_text_to_print(text)}\n\nhas detected the entities:\n")
  
  tokens_list = list(doc)    # Cada element de la llista és un objecte Token de spaCy
  
  print (f"{'Entity (ent.text)':<30}  Type  Tok_Start  Tok_End  Chr_Start  Chr_End  {'Entities (text string)':<30}  {'Entities (list Token)'}  ")
  print (f"{'=================':<30}  ====  =========  =======  =========  =======  {'======================':<30}  {'====================='}   ")
  for ent in doc.ents:
      print (f"{ent.text:<30}  {ent.label_:<4}  {ent.start:>9}  {ent.end:>7}  {ent.start_char:>9}  {ent.end_char:>7}  {text[ent.start_char:ent.end_char]:<30}  {tokens_list[ent.start:ent.end]}")
  print ("\n") 
  return doc   

In [ ]:
def load_entities():
  """Read entities for building KB

    Parameters: --

    Returns: 
      names (dict): Dictionary of entities: Keys: Qid; values: entity names.
      descriptions (dict): Diccionari of entities: Keys: Qis; values: entity descriptions.
  """
  entities_loc = Path.cwd() / "entitats_nel" / "entitats_KB.csv"

  names        = dict()
  descriptions = dict()
  with entities_loc.open("r", encoding="utf8") as csvfile:
    csvreader = csv.reader(csvfile, delimiter=",")
    next (csvreader)         # Saltar la capçalera
    for row in csvreader:
      qid, name, desc   = row[0], row[1], row[2]
      names[qid]        = name
      descriptions[qid] = desc
  return names, descriptions

In [ ]:
# Llegir el fitxer d'entitats que crearan la KB del sistema
name_dict, desc_dict = load_entities()

# Print per comprovació
print(*[f"{QID} -> entity name: '{name_dict[QID]}', description: '{desc_dict[QID]}'" for QID in name_dict.keys()], sep="\n")

Ara instanciem un objecte de tipus KnowledgeBase, especificant la dimensionalitat dels vectors d'entitats i el vocabulari:

In [ ]:
# Instanciar KB
kb = KnowledgeBase(vocab=nlp.vocab, entity_vector_length=300)

**Carregar la KB de spaCy** 

Cada entitat es carrega a la KB com:

*   Clau de l'entitat. En aquest exemple, l'identificador QID de Wikidata.
*   El 'doc embedding' o 'entity embedding' de la descripció de cada entitat. L' 'entity embedding' és la mitjana dels 'word embedding' de les paraules que figuren a la descripció. Condensa el significat de l'entitat en un únic vector (300D en aquest exemple).  


In [ ]:
# Carregar la KB de spaCy amb el qid i el doc embedding de la descripció de l'entitat
for qid, desc in desc_dict.items():
    desc_doc = nlp(desc)       
    desc_enc = desc_doc.vector
    kb.add_entity(entity=qid, entity_vector=desc_enc, freq=342)   # 342 is an arbitrary value here


**Afegir àlies**:  

Els àlies corresponents a entitats o grups d'entitats s'afegeixen a la KB. En aquest exemple la generació d'àlies es realitza manualment a partir de dues consideracions, són àlies:    

1.   Per a cada QID, un àlies amb el nom de l'entitat del QID.
2.   Un àlies genèric 'Piqué' per a les tres entitats exemple.  

La generalització d'aquesta acció depèn d'obtenir el màxim nombre possible d'àlies de les entitats del domini específic d'estudi a partir de fonts externes.  
Addicionalment, el model spaCy té en compte la probabilitat a priori d'una entitat donat un àlies. En un cas general, cal tenir en compte aquesta possibilitat (segons freqüències o altres criteris). 

Primer afegim el nom de cada entitat com a àlies, amb una probabilitat prèvia del 100%:

In [ ]:
# Àlies: nom de l'entitat per a cada Qid. Per a aquest cas, es considera 'probabilitat entitat donat àlies': 100%
for qid, name in name_dict.items():
    print(qid, name)
    kb.add_alias(alias=name, entities=[qid], probabilities=[1])   # 100% prior probability P(entity|alias)

També afegim 'Piqué' com a àlies per les 3 entitats, amb una probabilitat de 30% cadascuna:

In [ ]:
# Àlies: token genèric 'Piqué' per a cada QID de l'exemple. Es considera 'probabilitat entitat donat àlies': 30% cadascuna de les 3 entitats exemple.
qids = name_dict.keys()
probs = [0.3 for qid in qids]
kb.add_alias(alias="Piqué", entities=qids, probabilities=probs)  # sum([probs]) should be <= 1 !

In [ ]:
# Print de la KB creada
print(f"KB entities: {kb.get_entity_strings()}")
print(f"KB alias: {kb.get_alias_strings()}")

In [ ]:
# Un cop creada la KB
# Mètodes per conèixer, donat un token o tokens, quins candidats hi ha a la KB
print(f"Candidates of the mention 'Gerard Piqué': {[c.entity_ for c in kb.get_alias_candidates('Gerard Piqué')]}")
print(f"Candidates of the mention 'Piqué': {[c.entity_ for c in kb.get_alias_candidates('Piqué')]}")
print(f"Candidates of the mention 'UOC': {[c.entity_ for c in kb.get_alias_candidates('UOC')]}")

In [ ]:
!pwd

**Desar la KB creada**


In [ ]:
kb.to_disk("../tmp/nel_kb")    # Posterior recuperació utilitzar: kb.from_disk(Path)

##3.2 Creació fitxers d'entrenament i test

Per a realitzar l'entrenament s'ha de disposar de dades anotades. Al notebook sobre NER expliquem com realitzar anotacions amb anotadors externs i la seva conversió a format spaCy. Per a aquest exercici s'han anotat manualment trenta oracions, deu per a cada entitat exemple, en el format spaCy, incorporant-hi els camps necessaris per a 'Entity Linking'.

**Dades anotades, gold reference o ground-truth**

In [ ]:
# Exemple de suport per a anotar manualment
text = "Piqué obtingué un premi per la millor cobertura periodística sobre la renúncia de Benet XVI"
get_ents_to_print(nlp, text)

In [ ]:
dataset = [
           ("Piqué va pujar a atacar i va marcar un gol de cap.", {"links":{(0,5):{"Q17507": 1.0}}, "entities":[(0,5,"PER")]}),
           ("Quan l'entrenador va lliurar l'alineació, tots vam observar que Piqué no jugaria.", {"links":{(64,69):{"Q17507": 1.0}}, "entities":[(64,69,"PER")]}),
           ("En el partit contra el Llevant, Piqué va estar a punt de marcar, mentre que J. Alba no va jugar perquè estava lesionat.", {"links":{(32,37):{"Q17507": 1.0}}, "entities":[(32,37,"PER")]}),
           ("En acabar el partit, Piqué va declarar: 'No porto la samarreta del Barça per quedar tercer o quart'.", {"links":{(21,26):{"Q17507": 1.0}}, "entities":[(21,26,"PER")]}),
           ("El Barça ha demanat penal al minut 94 sobre Piqué, però l'àrbitre no l'ha xiulat.", {"links":{(44,49):{"Q17507": 1.0}}, "entities":[(44,49,"PER")]}),
           ("El Barça aconsegueix tres punts balsàmics contra el Dinamo de Kiev gràcies a un gol de Piqué.", {"links":{(87,92):{"Q17507": 1.0}}, "entities":[(87,92,"PER")]}),
           ("Mentre entrenava, Piqué va relliscar i es va torçar el peu", {"links":{(18,23):{"Q17507": 1.0}}, "entities":[(18,23,"PER")]}),
           ("Ha estat un partit on el davanter centre mai no ha pogut driblar en Piqué.", {"links":{(68,73):{"Q17507": 1.0}}, "entities":[(68,73,"PER")]}),
           ("Piqué no va jugar bé, la defensa es va empassar dos gols i el Barça va perdre 0-3.", {"links":{(0,5):{"Q17507": 1.0}}, "entities":[(0,5,"PER")]}),
           ("Piqué va jugar, com a titular, al partit de la Supercopa d'Europa 2015, a Tbilissi, en què el Barça va guanyar el Sevilla CF per 5 a 4.", {"links":{(0,5):{"Q17507": 1.0}}, "entities":[(0,5,"PER")]}),
           ("En Piqué va ser membre de la Junta directiva del Cercle d'Economia de Barcelona (1989-1995) i en fou el seu president.", {"links":{(3,8):{"Q1366326": 1.0}}, "entities":[(3,8,"PER")]}),
           ("L'any 1996, després de la victòria del Partit Popular en les eleccions generals, Piqué fou escollit pel nou president del Govern ministre d'Indústria i Energia, com a independent.", {"links":{(81,86):{"Q1366326": 1.0}}, "entities":[(81,86,"PER")]}),
           ("Piqué fou nomenat portaveu del Govern d'Espanya després del relleu de Miguel Ángel Rodríguez.", {"links":{(0,5):{"Q1366326": 1.0}}, "entities":[(0,5,"PER")]}),
           ("Quan era portaveu del Govern, Piqué va compaginar aquest càrrec amb la seva tasca de ministre d'Indústria fins al final de la VI legislatura.", {"links":{(30,35):{"Q1366326": 1.0}}, "entities":[(30,35,"PER")]}),
           ("Després de la victòria del Partit Popular, Piqué fou nomenat ministre d'Exteriors.", {"links":{(43,48):{"Q1366326": 1.0}}, "entities":[(43,48,"PER")]}),
           ("Piqué fou rellevat com a ministre d'Exteriors a la primera remodelació del segon govern d'Aznar", {"links":{(0,5):{"Q1366326": 1.0}}, "entities":[(0,5,"PER")]}),
           ("El ministre Piqué no va poder signar el decret perquè no tenia el vistiplau de l'assessor jurídic.", {"links":{(12,17):{"Q1366326": 1.0}}, "entities":[(12,17,"PER")]}),
           ("El 2003 Piqué abandonà les seves tasques al Govern d'Espanya per esdevenir candidat del Partit Popular de Catalunya (PPC) en les eleccions al Parlament de Catalunya", {"links":{(8,13):{"Q1366326": 1.0}}, "entities":[(8,13,"PER")]}),
           ("El 2007 Piqué presentà la dimissió irrevocable com a president del Partit Popular de Catalunya", {"links":{(8,13):{"Q1366326": 1.0}}, "entities":[(8,13,"PER")]}),
           ("Piqué va ser president de l'aerolínia de baix cost Vueling.", {"links":{(0,5):{"Q1366326": 1.0}}, "entities":[(0,5,"PER")]}),
           ("Piqué, periodista italiana nascuda a l'Argentina, és corresponsal al Vaticà pel diari La Nación.", {"links":{(0,5):{"Q21074333": 1.0}}, "entities":[(0,5,"PER")]}),
           ("Des del 1999 Piqué és corresponsal al Vaticà, un any en què ja entrevistà el que seria anys després el papa Francesc.", {"links":{(13,18):{"Q21074333": 1.0}}, "entities":[(13,18,"PER")]}),
           ("El 2013 Piqué va escriure una biografia del papa al llibre 'Francesc: vida i revolució'.", {"links":{(8,13):{"Q21074333": 1.0}}, "entities":[(8,13,"PER")]}),
           ("La pel·lícula 'Francisco: el padre Jorge' es basa en el llibre de la periodista Piqué.", {"links":{(80,85):{"Q21074333": 1.0}}, "entities":[(80,85,"PER")]}),
           ("El 2016 Piqué va escriure l'article a la Nación: Click to Pray, la app para rezar con el papa Francisco.", {"links":{(8,13):{"Q21074333": 1.0}}, "entities":[(8,13,"PER")]}),
           ("Piqué és llicenciada en Ciències Polítiques, especialitzada en Relacions Internacionals.", {"links":{(0,5):{"Q21074333": 1.0}}, "entities":[(0,5,"PER")]}),
           ("De 1992 a 1996, Piqué va ser redactora i editora del diario 'La Razón'.", {"links":{(16,21):{"Q21074333": 1.0}}, "entities":[(16,21,"PER")]}),
           ("El 1990, Piqué va ser becada per l'agència italiana de notícies ANSA, a Buenos Aires.", {"links":{(9,14):{"Q21074333": 1.0}}, "entities":[(9,14,"PER")]}),
           ("Piqué ha estat corresponsal de guerra a països com ara Afganistan, Irak, Líbia o Egipte.", {"links":{(0,5):{"Q21074333": 1.0}}, "entities":[(0,5,"PER")]}),
           ("Piqué obtingué un premi per la millor cobertura periodística sobre la renúncia de Benet XVI.", {"links":{(0,5):{"Q21074333": 1.0}}, "entities":[(0,5,"PER")]})
           ]
# Nota: Informació de detall extreta de Viquipèdia i/o elaboració pròpia          

In [ ]:
# Print per comprovació
print(*[f"Text ->'{text}'. Annotation: '{annot}'" for text, annot in dataset], sep="\n")

**Creació dels fitxers d'entrenament i de test**

El 'dataset' es fracciona en fitxers d'entrenament i de test amb un 80% i un 20% respectivament de les oracions exemple de cada QID (estratificació per QID). A continuació s'obté el fitxer d'entrenament final convertint cada text d'oració d'entrenament amb les seves anotacions en objectes 'Example' de spaCy. 

**Nota**: Tot i que per a aquest exercici els casos de prova s'han generat amb un únic qid per a cada registre i aquest qid pertany al KB (tret d'error), a continuació es considera el cas general, contemplant casos en què no existeix qid del KB (l'oració no passaria ni a entrenament ni a test), o bé que n'hi ha més d'un (si n'hi hagués més d'un, es consideraria que el registre té associat com a qid el primer)



In [ ]:
# Fraccionar el dataset en fitxer d'entrenament i test (80%, 20%), estratificat per qid. 

# Pas 1. Assignar un 'qid' a cada registre.
xs = []
ys = []            
# Cada registre de dataset que tingui un qid vàlid s'afegeix a 'xs' i el seu primer qid vàlid a 'ys'.
for index, (text, annot) in enumerate(dataset):
  y = []
  for span, links_dict in annot["links"].items():
    for link, value in links_dict.items():
      if link in qids and value:
        y.append(link)
  if len(y) != 0:
    xs.append(dataset[index])  
    ys.append(y[0])   

# Pas 2. Fraccionar 'xs' en entrenament i test, estratificat per 'ys' (qid)
train_dataset,test_dataset = train_test_split(xs,test_size=0.2,stratify=ys)

# Per comprovació
print (f"Number of sentences. Total: {len(dataset)}. Number of training sentences: {len(train_dataset)}. Number of test sentences: {len(test_dataset)}")
print (*[f"Test sentence: text -> '{text}'\t annotation -> {annot}" for text, annot in test_dataset], sep = "\n")

In [ ]:
# Transformar fitxer d'entrenament amb textos i anotacions a fitxer d'entrenament però amb objectes 'Example' de spaCy
# Nota 1: L'objecte 'Example' de spaCy conté tant les prediccions del model com les anotacions manuals (gold) i això permet entrenar el model Entity Linker de spaCy. 
# Nota 2: Donat que el model Entity Linker de spaCy utilitza el context de la menció (que es compara amb el word embedding de les entitats del KB), l'anotació manual de l'exemple es passa pel component 'sentencizer' per a obtenir les frases de l'exemple.

TRAIN_EXAMPLES = []
if "sentencizer" not in nlp.pipe_names:
  nlp.add_pipe("sentencizer")

sentencizer = nlp.get_pipe("sentencizer")
for text, annotation in train_dataset:
  example = Example.from_dict(nlp.make_doc(text), annotation)
  example.reference = sentencizer(example.reference)
  TRAIN_EXAMPLES.append(example)
print ("'train_dataset' has been transformed to 'train_examples', a dataset with spaCy 'example' objects")

##3.3 Entrenament del model

Es mostra a continuació com entrenar el model spaCy per a Entity Linking:

In [ ]:
# Afegir el component 'entity_linker' al pipe del model 'nlp'
entity_linker = nlp.add_pipe("entity_linker", config={"incl_prior": False}, last=True)

In [ ]:
# Inicialitzar el component entity_linker amb la kb creada i desada i alguns exemples d'entrenament que necessita entity_linker per a inicialitzar-se
entity_linker.initialize(get_examples=lambda: TRAIN_EXAMPLES, kb_loader=load_kb("../tmp/nel_kb"))     # Vegeu documentació: https://spacy.io/api/entitylinker#initialize

In [ ]:
# Entrenament del model
with nlp.select_pipes(enable=["entity_linker"]):   # train only the entity_linker
    optimizer = nlp.resume_training()
    for itn in range(1000):   
        random.shuffle(TRAIN_EXAMPLES)
        batches = minibatch(TRAIN_EXAMPLES, size=compounding(4.0, 32.0, 1.001))  # increasing batch sizes
        losses = {}
        for batch in batches:
            nlp.update(
                batch,   
                drop=0.2,      # prevent overfitting
                losses=losses,
                sgd=optimizer,
            )
        if itn % 50 == 0:
            print(itn, "Losses", losses)   # print the training loss
print(itn, "Losses", losses)

In [ ]:
# Exemple de predicció
text = "Piqué abandonà el Govern central per a presentar-se com a candidat del Partit Popular de Catalunya al Parlament."
doc = nlp(text)
for ent in doc.ents:
  url = "https://www.wikidata.org/wiki/"+ent.kb_id_ if ent.kb_id_ != "NIL" else "Entitat no desambiguada" 
  print(f"Mention: {ent.text}. Entity type: {ent.label_}. wikidata QID: {ent.kb_id_}. wikidata address: {url}")

In [ ]:
# Comprovació amb el fitxer de test.
for text, true_annot in test_dataset:
  print(text)
  print(f"Gold annotation: {true_annot}")
  doc = nlp(text)  # to make this more efficient, you can use nlp.pipe() just once for all the texts
  for ent in doc.ents:
    if ent.text == "Piqué":           # Llistar només l'entitat objecte d'aquest exemple
      url = "https://www.wikidata.org/wiki/"+ent.kb_id_ if ent.kb_id_ != "NIL" else "Entitat no desambiguada" 
      print(f"Mention: {ent.text}. Entity type: {ent.label_}. Wikidata QID: {ent.kb_id_}. Wikidata address: {url}")
  print()

In [ ]:
# Exemple de predicció
#text = "L'entrenador va alinear Piqué en el partit de la Supercopa d'Europa 2015 i el Barça va guanyar el Sevilla CF per 5 a 4."
text = "Piqué va jugar com a titular en el partit de la Supercopa d'Europa 2015 en què el Barça va guanyar el Sevilla CF per 5 a 4"
doc = nlp(text)
for ent in doc.ents:
  url = "https://www.wikidata.org/wiki/"+ent.kb_id_ if ent.kb_id_ != "NIL" else "Entitat no desambiguada" 
  print(f"Mention: {ent.text}.\tEntity type: {ent.label_}.\tWikidata QID: {ent.kb_id_}.\tWikidata address: {url}")

Donat el nombre tan reduït de casos d'entrenament (vuit per cada QID), no es poden extreure conclusions dels resultats de l'exercici. Aquests poden variar d'una execució a altra, ja que el fitxer d'entrenament se selecciona aleatòriament d'entre les dades inicials. En la major part de les execucions aquest exercici prediu correctament pel cap baix quatre dels sis exemples de test. 

Tot i que els resultats no són significatius, l'exercici mostra aspectes rellevants de la resolució NEL amb aquest tipus de mètodes, basats en la similitud entre el 'doc embedding' de la descripció de l'entitat i el 'doc embedding' del context de la menció:
* La importància de la descripció de les entitats del KB i la identificació d'àlies possibles de cada entitat.
* Les prediccions incorrectes acostumen a tenir un context ambigu o almenys susceptible de ser compartit entre diverses entitats d'un mateix àlies.

## 3.4 Exercicis

1. Cerca tres entitats que tinguin en comú un mateix àlies i redacta deu frases per a cada àlies-entitat. Crea la KB d'aquestes tres entitats. Segueix els passos descrits del procés NEL amb spaCy d'aquesta secció.
2. Donada una frase per a la qual s'ha realitzat una predicció incorrecta d'un àlies, altera gradualment el context de la menció, amb modificacions 'lleus' fins que el model la predigui correctament. Hi ha molta diferència entre la frase inicial predita incorrectament i la frase final predita correctament? La frase inicial era ambígua i el context del seu àlies podia considerar-se en l'àmbit de les dues definicions de l'entitat predita i la real? Els canvis han acostat el context de la menció a la definició de l'entitat associada real o ground-truth? 

#<a name="recursos"></a>4 Recursos

- Sistemes similars a DBPedia Spotlight:
  * [Wikifier](http://www.wikifier.org/). Utilitza com a KB `Wikipedia` i `Wikidata`. Proporciona eina [demo O/L](http://www.wikifier.org/) i [API](http://www.wikifier.org/info.html) d'accés.
  *   [Babelfy](http://babelfy.org/about). Utilitza una KB propia [Babelnet](https://babelnet.org/) (agrupa Wikipedia i [altres KB](https://babelnet.org/about) com ara wordnet, Geonames...) Proporciona eina [demo O/L](http://babelfy.org/) i [API](http://babelfy.org/guide) d'accés.

- [Improving Efficiency and Accuracy in Multilingual Entity Extraction](https://www.semanticscholar.org/paper/Improving-efficiency-and-accuracy-in-multilingual-Daiber-Jakob/a97fe719f2f1e62169320e1c45e3f96e094ccd19): Un article del 2013 que descriu una implementació de DBPedia Spotlight com un pipeline que fa server un model probabilístic per la desambiguació.

- [Training a Custom Entity Linking with Spacy](https://www.youtube.com/watch?v=8u57WSXVpmw): tutorial amb format vídeo. El notebook per SpaCy v3 es troba [aquí](https://github.com/explosion/projects/blob/v3/tutorials/nel_emerson/notebooks/notebook_video.ipynb).  

- [Spacy Entity Linking](https://spacy.io/universe/project/video-spacy-irl-entity-linking): presentació de l'arquitectura del mòdul de Spacy d'Entity Linking.